# NiVScan : Konversi Format IOB2

---

| | |
|---|---|
| **Mata Kuliah** | COMP6885001 — Natural Language Processing |
| **Tujuan** | Konversi dataset CSV ke format IOB2 untuk fine-tuning Model B, C, D |
| **Runtime** | CPU (tidak butuh GPU) |
| **Input** | `split_train.csv`, `split_val.csv`, `split_test_manual.csv` |
| **Output** | `dataset_train_iob.json`, `dataset_val_iob.json`, `dataset_test_iob.json` |

---

## ⚠️ Baca Dulu!

**Prasyarat:** Step 0 (`master_split.ipynb`) harus sudah dijalankan.

### Apa itu format IOB2?

Model BERT tidak bisa langsung menerima kalimat utuh — dia butuh input **per token** dengan label masing-masing:

```
Kalimat : "Virus Nipah menyebabkan demam di Malaysia."

Token       Label
─────────────────────────
Virus     → B-DISEASE   (Beginning of DISEASE)
Nipah     → I-DISEASE   (Inside DISEASE)
menyebabkan → O         (Outside / bukan entitas)
demam     → B-SYMPTOM   (Beginning of SYMPTOM)
di        → O
Malaysia  → B-LOCATION  (Beginning of LOCATION)
.         → O
```

### Notebook ini dikerjakan oleh 1 orang, hasilnya dibagikan ke semua anggota tim.
File JSON yang dihasilkan **wajib sama** untuk Model B, C, dan D.

---
## 1. Install & Import

In [1]:
# Semua library sudah tersedia di Colab — tidak perlu install tambahan
import pandas as pd
import numpy as np
import json
import re
from collections import Counter

print('Library berhasil diimport!')

Library berhasil diimport!


---
## 2. Mount Google Drive

In [2]:
from google.colab import drive
drive.mount('/content/drive')
print('Google Drive berhasil terhubung!')

Mounted at /content/drive
Google Drive berhasil terhubung!


In [3]:
# ============================================================
# SESUAIKAN PATH INI dengan lokasi Drive grup kalian
# ============================================================
BASE_PATH = '/content/drive/MyDrive/Binus University/Semester 4/NLP/NLP-Group Project/Dataset'

# Input (dari Step 0)
PATH_TRAIN  = f'{BASE_PATH}/split_train.csv'
PATH_VAL    = f'{BASE_PATH}/split_val.csv'
PATH_TEST   = f'{BASE_PATH}/split_test_manual.csv'

# Output (untuk Model B, C, D)
PATH_TRAIN_IOB = f'{BASE_PATH}/dataset_train_iob.json'
PATH_VAL_IOB   = f'{BASE_PATH}/dataset_val_iob.json'
PATH_TEST_IOB  = f'{BASE_PATH}/dataset_test_iob.json'

print('Path Input:')
print(f'   {PATH_TRAIN}')
print(f'   {PATH_VAL}')
print(f'   {PATH_TEST}')
print()
print('Path Output:')
print(f'   {PATH_TRAIN_IOB}')
print(f'   {PATH_VAL_IOB}')
print(f'   {PATH_TEST_IOB}')

Path Input:
   /content/drive/MyDrive/Binus University/Semester 4/NLP/NLP-Group Project/Dataset/split_train.csv
   /content/drive/MyDrive/Binus University/Semester 4/NLP/NLP-Group Project/Dataset/split_val.csv
   /content/drive/MyDrive/Binus University/Semester 4/NLP/NLP-Group Project/Dataset/split_test_manual.csv

Path Output:
   /content/drive/MyDrive/Binus University/Semester 4/NLP/NLP-Group Project/Dataset/dataset_train_iob.json
   /content/drive/MyDrive/Binus University/Semester 4/NLP/NLP-Group Project/Dataset/dataset_val_iob.json
   /content/drive/MyDrive/Binus University/Semester 4/NLP/NLP-Group Project/Dataset/dataset_test_iob.json


---
## 3. Load Dataset

In [4]:
import os
for p in [PATH_TRAIN, PATH_VAL, PATH_TEST]:
    assert os.path.exists(p), f' File tidak ditemukan: {p}\n   → Jalankan Step 0 terlebih dahulu!'

df_train = pd.read_csv(PATH_TRAIN)
df_val   = pd.read_csv(PATH_VAL)
df_test  = pd.read_csv(PATH_TEST)

print('Dataset berhasil diload!')
print(f'   split_train.csv       : {len(df_train)} baris')
print(f'   split_val.csv         : {len(df_val)} baris')
print(f'   split_test_manual.csv : {len(df_test)} baris')
print()
print('Contoh data:')
print(df_train[['text_clean','DISEASE','SYMPTOM','LOCATION']].head(3).to_string())

Dataset berhasil diload!
   split_train.csv       : 574 baris
   split_val.csv         : 144 baris
   split_test_manual.csv : 179 baris

Contoh data:
                                                                                                                                       text_clean              DISEASE SYMPTOM                    LOCATION
0                                      No infection of humans or other species has been observed in Cambodia, Thailand, or Africa as of May 2018.                  NaN     NaN  Cambodia, Thailand, Africa
1  It is difficult to distinguish Nipah from other infectious diseases, or other causes of encephalitis or pneumonia, without laboratory testing.  Nipah, encephalitis     NaN                         NaN
2                                                                              Virus Nipah, Ancaman Zoonosis yang Perlu Diwaspadai - Kompaspedia.                Nipah     NaN                         NaN


---
## 4. Fungsi Konversi ke Format IOB2

Strategi pelabelan:
1. Tokenisasi kalimat dengan `split()` sederhana (whitespace)
2. Untuk setiap entitas (DISEASE/SYMPTOM/LOCATION), cari kemunculannya di kalimat
3. Token pertama entitas → `B-LABEL`, token berikutnya → `I-LABEL`
4. Token yang bukan entitas → `O`
5. Jika ada konflik (token masuk ke dua entitas), prioritas: DISEASE > SYMPTOM > LOCATION

In [5]:
def parse_entities(val):
    """Parse comma-separated entity string menjadi list."""
    if pd.isna(val) or str(val).strip() == '':
        return []
    return [e.strip() for e in str(val).split(',') if e.strip()]


def tokenize(text):
    """Tokenisasi sederhana — split by whitespace, pertahankan tanda baca."""
    # Pisahkan tanda baca dari kata
    tokens = re.findall(r"\w+(?:'\w+)*|[^\w\s]", text)
    return tokens


def find_entity_spans(tokens, entity_phrase):
    """
    Cari posisi token (start, end) dari sebuah entity phrase dalam list tokens.
    Return list of (start_idx, end_idx) — end_idx exclusive.
    """
    entity_tokens = tokenize(entity_phrase.lower())
    n = len(entity_tokens)
    spans = []

    for i in range(len(tokens) - n + 1):
        window = [t.lower() for t in tokens[i:i+n]]
        if window == entity_tokens:
            spans.append((i, i + n))

    return spans


def convert_to_iob(row):
    """
    Konversi 1 baris DataFrame ke format IOB2.
    Return dict dengan 'tokens' dan 'ner_tags'.
    """
    text = str(row['text_clean'])
    tokens = tokenize(text)

    if not tokens:
        return None

    # Inisialisasi semua token sebagai 'O'
    labels = ['O'] * len(tokens)

    # Prioritas: DISEASE > SYMPTOM > LOCATION
    # Proses dari prioritas terendah ke tertinggi supaya yang lebih tinggi bisa overwrite
    entity_map = [
        ('LOCATION', parse_entities(row.get('LOCATION'))),
        ('SYMPTOM',  parse_entities(row.get('SYMPTOM'))),
        ('DISEASE',  parse_entities(row.get('DISEASE'))),
    ]

    for label, entities in entity_map:
        for entity in entities:
            if not entity:
                continue
            spans = find_entity_spans(tokens, entity)
            for start, end in spans:
                labels[start] = f'B-{label}'
                for i in range(start + 1, end):
                    labels[i] = f'I-{label}'

    return {
        'id': str(row.get('id', '')),
        'tokens': tokens,
        'ner_tags': labels
    }


print('Fungsi konversi IOB2 berhasil didefinisikan!')
print()

# Test dengan 1 contoh
test_row = pd.Series({
    'id': 'test',
    'text_clean': 'Virus Nipah menyebabkan demam di Malaysia.',
    'DISEASE': 'Virus Nipah',
    'SYMPTOM': 'demam',
    'LOCATION': 'Malaysia'
})
result = convert_to_iob(test_row)
print('Test konversi:')
print(f'   Kalimat  : {test_row["text_clean"]}')
print(f'   Tokens   : {result["tokens"]}')
print(f'   IOB Tags : {result["ner_tags"]}')
print()
for tok, tag in zip(result['tokens'], result['ner_tags']):
    print(f'   {tok:20s} → {tag}')

Fungsi konversi IOB2 berhasil didefinisikan!

Test konversi:
   Kalimat  : Virus Nipah menyebabkan demam di Malaysia.
   Tokens   : ['Virus', 'Nipah', 'menyebabkan', 'demam', 'di', 'Malaysia', '.']
   IOB Tags : ['B-DISEASE', 'I-DISEASE', 'O', 'B-SYMPTOM', 'O', 'B-LOCATION', 'O']

   Virus                → B-DISEASE
   Nipah                → I-DISEASE
   menyebabkan          → O
   demam                → B-SYMPTOM
   di                   → O
   Malaysia             → B-LOCATION
   .                    → O


---
## 5. Konversi Semua Split

In [6]:
def convert_dataframe(df, nama):
    """Konversi seluruh DataFrame ke list of IOB dicts."""
    results = []
    skipped = 0

    for _, row in df.iterrows():
        converted = convert_to_iob(row)
        if converted is not None:
            results.append(converted)
        else:
            skipped += 1

    print(f'{nama}: {len(results)} berhasil dikonversi, {skipped} dilewati (kosong)')
    return results


print('Mengkonversi dataset...')
train_iob = convert_dataframe(df_train, 'Train')
val_iob   = convert_dataframe(df_val,   'Val')
test_iob  = convert_dataframe(df_test,  'Test')
print()
print('Konversi selesai!')

Mengkonversi dataset...
Train: 574 berhasil dikonversi, 0 dilewati (kosong)
Val: 144 berhasil dikonversi, 0 dilewati (kosong)
Test: 179 berhasil dikonversi, 0 dilewati (kosong)

Konversi selesai!


---
## 6. Verifikasi Hasil Konversi

In [7]:
# Cek distribusi label di train set
all_labels = [tag for item in train_iob for tag in item['ner_tags']]
label_counts = Counter(all_labels)

print('Distribusi label di dataset_train_iob:')
total = sum(label_counts.values())
for label, count in sorted(label_counts.items()):
    print(f'   {label:15s} : {count:5d} token ({count/total*100:.2f}%)')
print(f'   {"TOTAL":15s} : {total:5d} token')
print()

# Cek berapa kalimat yang punya setidaknya 1 entitas
has_entity_train = sum(1 for item in train_iob if any(t != 'O' for t in item['ner_tags']))
has_entity_test  = sum(1 for item in test_iob  if any(t != 'O' for t in item['ner_tags']))

print(f'   Train: {has_entity_train}/{len(train_iob)} kalimat punya entitas ({has_entity_train/len(train_iob)*100:.1f}%)')
print(f'   Test : {has_entity_test}/{len(test_iob)} kalimat punya entitas ({has_entity_test/len(test_iob)*100:.1f}%)')

Distribusi label di dataset_train_iob:
   B-DISEASE       :   296 token (2.84%)
   B-LOCATION      :   250 token (2.40%)
   B-SYMPTOM       :    45 token (0.43%)
   I-DISEASE       :   121 token (1.16%)
   I-LOCATION      :    39 token (0.37%)
   I-SYMPTOM       :     5 token (0.05%)
   O               :  9669 token (92.75%)
   TOTAL           : 10425 token

   Train: 341/574 kalimat punya entitas (59.4%)
   Test : 118/179 kalimat punya entitas (65.9%)


In [8]:
# Tampilkan beberapa contoh hasil konversi untuk dicek manual
print('Contoh hasil konversi (5 kalimat pertama yang punya entitas):\n')
count = 0
for item in train_iob:
    if any(t != 'O' for t in item['ner_tags']):
        print(f'ID: {item["id"]}')
        for tok, tag in zip(item['tokens'], item['ner_tags']):
            marker = ' ←' if tag != 'O' else ''
            print(f'   {tok:25s} {tag}{marker}')
        print()
        count += 1
        if count >= 5:
            break

Contoh hasil konversi (5 kalimat pertama yang punya entitas):

ID: 749
   No                        O
   infection                 O
   of                        O
   humans                    O
   or                        O
   other                     O
   species                   O
   has                       O
   been                      O
   observed                  O
   in                        O
   Cambodia                  B-LOCATION ←
   ,                         O
   Thailand                  B-LOCATION ←
   ,                         O
   or                        O
   Africa                    B-LOCATION ←
   as                        O
   of                        O
   May                       O
   2018                      O
   .                         O

ID: 861
   It                        O
   is                        O
   difficult                 O
   to                        O
   distinguish               O
   Nipah                     B-DISEASE ←
   from  

---
## 7. Simpan ke Google Drive

In [10]:
def save_json(data, path):
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

save_json(train_iob, PATH_TRAIN_IOB)
save_json(val_iob,   PATH_VAL_IOB)
save_json(test_iob,  PATH_TEST_IOB)

print('Semua file IOB berhasil disimpan ke Google Drive!')
print(f'   dataset_train_iob.json : {len(train_iob)} kalimat')
print(f'   dataset_val_iob.json   : {len(val_iob)} kalimat')
print(f'   dataset_test_iob.json  : {len(test_iob)} kalimat')

Semua file IOB berhasil disimpan ke Google Drive!
   dataset_train_iob.json : 574 kalimat
   dataset_val_iob.json   : 144 kalimat
   dataset_test_iob.json  : 179 kalimat


---
## 8. Verifikasi Final — Reload & Cek

In [11]:
# Reload untuk memastikan file tersimpan dengan benar
def load_json(path):
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)

check_train = load_json(PATH_TRAIN_IOB)
check_val   = load_json(PATH_VAL_IOB)
check_test  = load_json(PATH_TEST_IOB)

print('Verifikasi final:')
print(f'   dataset_train_iob.json : {len(check_train)} kalimat')
print(f'   dataset_val_iob.json   : {len(check_val)} kalimat')
print(f'   dataset_test_iob.json  : {len(check_test)} kalimat')
print()

# Cek struktur 1 sampel
sample = check_train[0]
print('Struktur JSON (1 sampel dari train):')
print(f'   Keys    : {list(sample.keys())}')
print(f'   id      : {sample["id"]}')
print(f'   tokens  : {sample["tokens"][:5]}...')
print(f'   ner_tags: {sample["ner_tags"][:5]}...')

Verifikasi final:
   dataset_train_iob.json : 574 kalimat
   dataset_val_iob.json   : 144 kalimat
   dataset_test_iob.json  : 179 kalimat

Struktur JSON (1 sampel dari train):
   Keys    : ['id', 'tokens', 'ner_tags']
   id      : 749
   tokens  : ['No', 'infection', 'of', 'humans', 'or']...
   ner_tags: ['O', 'O', 'O', 'O', 'O']...


In [12]:
print('=' * 62)
print('STEP 1 SELESAI — Konversi IOB2 Berhasil!')
print('=' * 62)
print()
print('File yang tersedia di Drive untuk Model B, C, D:')
print(f'   dataset_train_iob.json → {len(check_train)} kalimat (fine-tuning)')
print(f'   dataset_val_iob.json   → {len(check_val)} kalimat (early stopping)')
print(f'   dataset_test_iob.json  → {len(check_test)} kalimat (evaluasi final)')
print()
print('Langkah selanjutnya (paralel, masing-masing anggota):')
print('   Kamu      → Model B: dslim/bert-base-NER          (butuh GPU)')
print('   Teman 1   → Model C: d4data/biomedical-ner-all    (butuh GPU)')
print('   Teman 2   → Model D: cahya/bert-base-indonesian-NER (butuh GPU)')
print()
print('   Semua model load file IOB yang SAMA dari Drive.')
print('   Evaluasi akhir semua model pakai dataset_test_iob.json.')
print()
print('JANGAN jalankan notebook ini lagi!')

STEP 1 SELESAI — Konversi IOB2 Berhasil!

File yang tersedia di Drive untuk Model B, C, D:
   dataset_train_iob.json → 574 kalimat (fine-tuning)
   dataset_val_iob.json   → 144 kalimat (early stopping)
   dataset_test_iob.json  → 179 kalimat (evaluasi final)

Langkah selanjutnya (paralel, masing-masing anggota):
   Kamu      → Model B: dslim/bert-base-NER          (butuh GPU)
   Teman 1   → Model C: d4data/biomedical-ner-all    (butuh GPU)
   Teman 2   → Model D: cahya/bert-base-indonesian-NER (butuh GPU)

   Semua model load file IOB yang SAMA dari Drive.
   Evaluasi akhir semua model pakai dataset_test_iob.json.

JANGAN jalankan notebook ini lagi!
